# LeetCode 1705 — Maximum Number of Eaten Apples bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-eaten-apples/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def eatenApples(self, apples: List[int], days: List[int]) -> int:
        ...
```

Lộ trình: **ghép cặp táo ↔ ngày** (matching hai phía — đáp án chuẩn cho input nhỏ) → **tham lam "ăn quả sắp hỏng nhất"** bằng quét tuyến tính (`O(n · T)`) → **min-heap theo ngày hỏng** (`O((n + T) log n)`) → **heap + nhảy cóc** sau ngày `n` (ăn cả lô một lần).

## 1. Mục tiêu học

1. Mô hình hóa bài toán thành **lập lịch**: mỗi quả táo là một "việc" có **khoảng thời gian hợp lệ** `[ngày mọc, ngày hỏng − 1]`, mỗi ngày làm được tối đa một việc.
2. Hiểu và chứng minh chiến lược **Earliest Deadline First (EDF)**: mỗi ngày ăn quả **hỏng sớm nhất** trong số quả còn ăn được.
3. Dùng **min-heap** lưu các lô táo `(ngày hỏng, số quả)` và **xóa lười** các lô đã hỏng.
4. Tối ưu pha sau ngày `n` bằng cách **ăn cả lô một lần** (nhảy thời gian), tránh mô phỏng từng ngày.
5. Kiểm chứng tham lam bằng **ghép cặp cực đại** (thuật toán Kuhn) trên input nhỏ.

## 2. Đề bài & ví dụ

Một cây táo đặc biệt ra quả trong `n` ngày. Ngày thứ `i` (từ 0), cây cho `apples[i]` quả, và các quả này sẽ **hỏng sau `days[i]` ngày**, tức là vào ngày `i + days[i]` chúng hỏng và không ăn được nữa. Những ngày cây không ra quả có `apples[i] == 0` và `days[i] == 0`.

Bạn ăn **tối đa một quả mỗi ngày**. Bạn có thể tiếp tục ăn **sau** `n` ngày đầu.

Cho `apples` và `days` cùng độ dài `n`, trả về **số quả nhiều nhất** có thể ăn.

**Ví dụ 1:** `apples = [1,2,3,5,2]`, `days = [3,2,1,4,2]` → `7`.
- Ngày 0 ăn quả mọc ngày 0. Ngày 1 ăn quả mọc ngày 1. Ngày 2 ăn quả mọc ngày 1 (quả mọc ngày 2 hỏng ngay sau ngày 2). Ngày 3 → 6 ăn quả mọc ngày 3.

**Ví dụ 2:** `apples = [3,0,0,0,0,2]`, `days = [3,0,0,0,0,2]` → `5`.
- Ngày 0 → 2 ăn quả ngày 0; ngày 3, 4 không có gì; ngày 5, 6 ăn quả ngày 5.

**Ràng buộc:**
- `n == apples.length == days.length`, `1 ≤ n ≤ 2·10⁴`
- `0 ≤ apples[i], days[i] ≤ 2·10⁴`
- `days[i] == 0` khi và chỉ khi `apples[i] == 0`.

In [1]:
import random
import time
import heapq
from typing import List

examples = [
    (([1, 2, 3, 5, 2], [3, 2, 1, 4, 2]), 7),
    (([3, 0, 0, 0, 0, 2], [3, 0, 0, 0, 0, 2]), 5),
]
for (apples, days), want in examples:
    print(f"apples={apples}, days={days} -> kỳ vọng {want}")

apples=[1, 2, 3, 5, 2], days=[3, 2, 1, 4, 2] -> kỳ vọng 7
apples=[3, 0, 0, 0, 0, 2], days=[3, 0, 0, 0, 0, 2] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1 Khoảng ngày ăn được của một lô

Lô mọc ngày `i` với hạn `d = days[i]` ăn được vào các ngày `i, i+1, …, i+d−1` (hỏng ở ngày `i+d`). Lưu **ngày hỏng** `rot = i + d`: lô còn ăn được ở ngày `t` ⇔ `t < rot`.

In [2]:
apples, days = [1, 2, 3, 5, 2], [3, 2, 1, 4, 2]
for i, (a, d) in enumerate(zip(apples, days)):
    if a:
        print(f"lô ngày {i}: {a} quả, ăn được các ngày {list(range(i, i + d))}, hỏng ở ngày {i + d}")

lô ngày 0: 1 quả, ăn được các ngày [0, 1, 2], hỏng ở ngày 3
lô ngày 1: 2 quả, ăn được các ngày [1, 2], hỏng ở ngày 3
lô ngày 2: 3 quả, ăn được các ngày [2], hỏng ở ngày 3
lô ngày 3: 5 quả, ăn được các ngày [3, 4, 5, 6], hỏng ở ngày 7
lô ngày 4: 2 quả, ăn được các ngày [4, 5], hỏng ở ngày 6


### 3.2 `heapq` — min-heap của tuple

Tuple được so sánh theo thứ tự từng thành phần ⇒ heap `(rot, count)` luôn cho lô **hỏng sớm nhất** ở `h[0]`. Muốn sửa số quả của lô ở đỉnh: `heappop` rồi `heappush` lại, hoặc lưu `list` có thể thay đổi `[rot, count]` và sửa tại chỗ phần tử đỉnh (không làm hỏng thứ tự heap vì khóa `rot` không đổi).

In [3]:
h = []
for rot, cnt in [(5, 2), (3, 1), (8, 4)]:
    heapq.heappush(h, [rot, cnt])
print("đỉnh:", h[0])
h[0][1] -= 1              # ăn một quả của lô đỉnh — khóa rot không đổi nên heap vẫn hợp lệ
print("sau khi ăn:", h[0])

đỉnh: [3, 1]
sau khi ăn: [3, 0]


### 3.3 Xóa lười (lazy deletion)

Lô đã hỏng (`rot ≤ t`) hoặc đã ăn hết (`count == 0`) không cần xóa ngay — chỉ cần bỏ khi nó nổi lên đỉnh heap. Vì heap sắp theo `rot`, nếu đỉnh còn tốt thì mọi lô hỏng đã bị lọc.

### 3.4 Ghép cặp hai phía (bipartite matching) — thuật toán Kuhn

Bên trái: từng quả táo. Bên phải: từng ngày. Cạnh nối quả với các ngày nó ăn được. Ghép cặp cực đại = số quả ăn được nhiều nhất. Kuhn: với mỗi đỉnh trái, DFS tìm **đường tăng** (augmenting path). Chậm (`O(V·E)`), chỉ dùng để kiểm chứng.

In [4]:
def kuhn(adj, n_right):
    match_r = [-1] * n_right
    def try_kuhn(u, seen):
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                if match_r[v] == -1 or try_kuhn(match_r[v], seen):
                    match_r[v] = u
                    return True
        return False
    return sum(try_kuhn(u, [False] * n_right) for u in range(len(adj)))

# 3 việc, 2 ngày: việc 0 làm ngày 0, việc 1 làm ngày 0 hoặc 1, việc 2 làm ngày 1
print(kuhn([[0], [0, 1], [1]], 2))

2


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 — vẽ các khoảng ăn được:

| lô (ngày mọc) | số quả | ăn được ngày | hỏng ở |
|---|---|---|---|
| 0 | 1 | 0, 1, 2 | 3 |
| 1 | 2 | 1, 2 | 3 |
| 2 | 3 | 2 | 3 |
| 3 | 5 | 3, 4, 5, 6 | 7 |
| 4 | 2 | 4, 5 | 6 |

Ngày 2 có ba lô cạnh tranh: lô 0, lô 1, lô 2 — **cả ba đều hỏng ở ngày 3**. Ngày 0 chỉ có lô 0; ngày 1 có lô 0 và lô 1. Dù chọn thế nào, ngày 0–2 ăn được 3 quả. Ngày 3–6: lô 3 (5 quả, hạn tới ngày 6) và lô 4 (2 quả, hạn tới ngày 5) — chỉ có 4 ngày ⇒ tối đa 4 quả. Tổng `3 + 4 = 7`.

**Trực giác:** khi phải chọn, hãy ăn quả **sắp hỏng nhất** — quả hạn xa vẫn còn cơ hội ăn sau; quả sắp hỏng thì không.

## 5. Cách 1 — Brute Force / baseline: ghép cặp cực đại quả ↔ ngày

**Ý tưởng:** mỗi quả là một đỉnh trái, mỗi ngày (từ 0 tới ngày hỏng muộn nhất) là một đỉnh phải; quả của lô `i` nối với các ngày `i..i+days[i]−1`. Đáp án = kích thước ghép cặp cực đại. Đúng theo định nghĩa bài toán (mỗi ngày ≤ 1 quả, mỗi quả ăn ≤ 1 lần, đúng hạn), không cần tin vào tham lam.

In [5]:
def eaten_matching(apples, days):
    T = max((i + d for i, d in enumerate(days)), default=0)
    adj = []
    for i, (a, d) in enumerate(zip(apples, days)):
        for _ in range(a):
            adj.append(list(range(i, i + d)))
    return kuhn(adj, T)

for (apples, days), want in examples:
    print(eaten_matching(apples, days), "kỳ vọng", want)

7 kỳ vọng 7
5 kỳ vọng 5


**Độ phức tạp:** số đỉnh trái `A = Σ apples` (tới `4·10⁸`!), số cạnh `Σ apples[i]·days[i]` ⇒ hoàn toàn không khả thi với ràng buộc thật. Kuhn `O(A · E)`.

**Nút thắt:** không khai thác cấu trúc **khoảng** của các cạnh. Với khoảng, tham lam theo hạn chót là tối ưu (mục 6).

In [6]:
random.seed(0)
for n in [6, 10, 14]:
    apples = [random.randint(0, 4) for _ in range(n)]
    days = [random.randint(1, 5) if a else 0 for a in apples]
    t0 = time.perf_counter(); r = eaten_matching(apples, days)
    print(f"n={n}, tổng quả={sum(apples)}: {r} quả, {time.perf_counter() - t0:.4f}s")

n=6, tổng quả=15: 10 quả, 0.0000s
n=10, tổng quả=23: 11 quả, 0.0000s
n=14, tổng quả=33: 15 quả, 0.0000s


## 6. Ý tưởng thuật toán / bước đột phá

**Quy tắc EDF:** *Mỗi ngày `t`, nếu còn quả ăn được, ăn một quả thuộc lô có **ngày hỏng nhỏ nhất**.*

**Lập luận đổi chỗ (exchange argument):** giả sử một lịch tối ưu `OPT` ở ngày `t` ăn quả `x` có hạn `rot_x`, trong khi có quả `y` ăn được với `rot_y ≤ rot_x`.
- Nếu `OPT` không bao giờ ăn `y`: thay `x` bằng `y` ở ngày `t` — vẫn hợp lệ, số quả không đổi.
- Nếu `OPT` ăn `y` ở ngày `t' > t`: đổi chỗ `x` và `y`. `y` ăn ở `t` (hợp lệ vì `y` đã mọc và `t < t' < rot_y`); `x` ăn ở `t'` — hợp lệ vì `t' < rot_y ≤ rot_x` và `x` đã mọc trước `t`. Số quả không đổi.

Lặp lại ⇒ tồn tại lịch tối ưu trùng với EDF ở mọi ngày.

**Thêm một ý:** ngày nào còn quả thì **luôn ăn** (bỏ trống không bao giờ có lợi — quả để dành có thể ăn sau, nhưng ngày trống thì mất luôn).

**Invariant cho heap:** ở đầu ngày `t` (sau khi thêm lô mới và lọc), heap chứa đúng các lô còn quả và chưa hỏng; đỉnh là lô hỏng sớm nhất.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — EDF bằng quét tuyến tính

Từng đoạn:
1. `rem[i]` = số quả còn lại của lô `i`.
2. Mỗi ngày `t = 0, 1, …` tới ngày hỏng muộn nhất: quét mọi lô `i ≤ min(t, n−1)` còn quả và `i + days[i] > t`, chọn lô hỏng sớm nhất.
3. Ăn một quả.

In [7]:
def eaten_scan(apples, days):
    n = len(apples)
    rem = list(apples)
    T = max(i + d for i, d in enumerate(days))
    eaten = 0
    for t in range(T):
        best = -1
        for i in range(min(t, n - 1) + 1):
            if rem[i] > 0 and i + days[i] > t:
                if best == -1 or i + days[i] < best_rot:
                    best, best_rot = i, i + days[i]
        if best != -1:
            rem[best] -= 1
            eaten += 1
    return eaten

for (apples, days), want in examples:
    print(eaten_scan(apples, days), "kỳ vọng", want)

7 kỳ vọng 7
5 kỳ vọng 5


**Độ phức tạp:** `T ≤ n + max(days) ≤ 4·10⁴` ngày, mỗi ngày quét `O(n)` lô ⇒ `O(n · T) ≈ 8·10⁸` worst — quá chậm trong Python, nhưng đúng thuật toán.

### 7.2 Cách 3 — EDF bằng min-heap, mô phỏng từng ngày

Từng đoạn:
1. Ngày `t < n`: nếu `apples[t] > 0`, đẩy `[t + days[t], apples[t]]` vào heap.
2. Lọc đỉnh: `while h and (h[0][0] ≤ t or h[0][1] == 0): heappop`.
3. Nếu heap không rỗng: ăn một quả của đỉnh (`h[0][1] −= 1`), `eaten += 1`.
4. Tiếp tục cho tới khi `t ≥ n` **và** heap rỗng.

In [8]:
def eaten_heap(apples, days):
    n = len(apples)
    h = []
    eaten = 0
    t = 0
    while t < n or h:
        if t < n and apples[t] > 0:
            heapq.heappush(h, [t + days[t], apples[t]])
        while h and (h[0][0] <= t or h[0][1] == 0):
            heapq.heappop(h)
        if h:
            h[0][1] -= 1
            eaten += 1
        t += 1
    return eaten

for (apples, days), want in examples:
    print(eaten_heap(apples, days), "kỳ vọng", want)

7 kỳ vọng 7
5 kỳ vọng 5


**Độ phức tạp:** số ngày mô phỏng ≤ `n + max(days)`; mỗi lô push/pop 1 lần ⇒ Time `O((n + D) + n log n)` với `D = max(days)`, Space `O(n)`.

### 7.3 Cách 4 — Heap + nhảy cóc sau ngày `n`

Sau ngày `n − 1` không còn lô mới. Khi đó lô đỉnh (hỏng sớm nhất, ở ngày `rot`, còn `c` quả) sẽ được ăn **liên tục** cho đến khi hết quả hoặc hỏng: ăn `k = min(c, rot − t)` quả, nhảy `t += k`. Số bước sau ngày `n` chỉ còn `O(số lô)`.

In [9]:
def eaten_heap_jump(apples, days):
    n = len(apples)
    h = []
    eaten = 0
    for t in range(n):
        if apples[t] > 0:
            heapq.heappush(h, (t + days[t], apples[t]))
        while h and h[0][0] <= t:
            heapq.heappop(h)
        if h:
            rot, c = heapq.heappop(h)
            eaten += 1
            if c > 1:
                heapq.heappush(h, (rot, c - 1))
    t = n
    while h:
        rot, c = heapq.heappop(h)
        if rot <= t:
            continue
        k = min(c, rot - t)
        eaten += k
        t += k
    return eaten

for (apples, days), want in examples:
    print(eaten_heap_jump(apples, days), "kỳ vọng", want)

7 kỳ vọng 7
5 kỳ vọng 5


**Độ phức tạp:** pha 1 có `n` ngày, mỗi ngày `O(log n)`; pha 2 mỗi lô bị pop một lần ⇒ Time `O(n log n)`, Space `O(n)`. Không phụ thuộc `max(days)`.

Vì sao pha 2 đúng: sau ngày `n`, EDF luôn ăn lô hỏng sớm nhất; ăn quả của lô đó không làm thay đổi thứ tự hạn của các lô khác, nên ta cứ ăn nó cho đến khi hết/hỏng — đúng những ngày EDF ăn từng quả một.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 `apples = [1,2,3,5,2]`, `days = [3,2,1,4,2]`:

| ngày | thêm lô | lọc bỏ | heap sau khi ăn (rot:còn) | ăn | tổng |
|---|---|---|---|---|---|
| 0 | [3,1] | — | [3:0] | lô hỏng ngày 3 | 1 |
| 1 | [3,2] | [3:0] (hết) | [3:1] | lô hỏng ngày 3 | 2 |
| 2 | [3,3] | — | [3:0], [3:3] | một lô hỏng ngày 3 | 3 |
| 3 | [7,5] | các lô rot = 3 | [7:4] | lô hỏng ngày 7 | 4 |
| 4 | [6,2] | — | [6:1], [7:4] | lô hỏng ngày 6 | 5 |
| 5 | — | — | [6:0], [7:4] | lô hỏng ngày 6 | 6 |
| 6 | — | [6:0] | [7:3] | lô hỏng ngày 7 | 7 |
| 7 | — | [7:3] (hỏng) | rỗng | — | 7 |

In [10]:
def trace_heap(apples, days):
    n = len(apples); h = []; eaten = 0; t = 0
    while t < n or h:
        added = None
        if t < n and apples[t] > 0:
            heapq.heappush(h, [t + days[t], apples[t]]); added = (t + days[t], apples[t])
        dropped = []
        while h and (h[0][0] <= t or h[0][1] == 0):
            dropped.append(tuple(heapq.heappop(h)))
        ate = None
        if h:
            h[0][1] -= 1; eaten += 1; ate = h[0][0]
        state = sorted(tuple(x) for x in h)
        print(f"ngày {t}: thêm {added or '-'}, bỏ {dropped or '-'}, ăn lô hỏng ngày {ate if ate is not None else '-'}, "
              f"heap(rot,còn)={state}, tổng={eaten}")
        t += 1
    return eaten

trace_heap([1, 2, 3, 5, 2], [3, 2, 1, 4, 2])

ngày 0: thêm (3, 1), bỏ -, ăn lô hỏng ngày 3, heap(rot,còn)=[(3, 0)], tổng=1
ngày 1: thêm (3, 2), bỏ [(3, 0)], ăn lô hỏng ngày 3, heap(rot,còn)=[(3, 1)], tổng=2
ngày 2: thêm (3, 3), bỏ -, ăn lô hỏng ngày 3, heap(rot,còn)=[(3, 0), (3, 3)], tổng=3
ngày 3: thêm (7, 5), bỏ [(3, 0), (3, 3)], ăn lô hỏng ngày 7, heap(rot,còn)=[(7, 4)], tổng=4
ngày 4: thêm (6, 2), bỏ -, ăn lô hỏng ngày 6, heap(rot,còn)=[(6, 1), (7, 4)], tổng=5
ngày 5: thêm -, bỏ -, ăn lô hỏng ngày 6, heap(rot,còn)=[(6, 0), (7, 4)], tổng=6
ngày 6: thêm -, bỏ [(6, 0)], ăn lô hỏng ngày 7, heap(rot,còn)=[(7, 3)], tổng=7
ngày 7: thêm -, bỏ [(7, 3)], ăn lô hỏng ngày -, heap(rot,còn)=[], tổng=7


7

In [11]:
def trace_jump(apples, days):
    n = len(apples); h = []; eaten = 0
    for t in range(n):
        if apples[t] > 0:
            heapq.heappush(h, (t + days[t], apples[t]))
        while h and h[0][0] <= t:
            heapq.heappop(h)
        if h:
            rot, c = heapq.heappop(h); eaten += 1
            if c > 1: heapq.heappush(h, (rot, c - 1))
    print(f"hết pha 1 (ngày 0..{n - 1}): đã ăn {eaten}, heap = {sorted(h)}")
    t = n
    while h:
        rot, c = heapq.heappop(h)
        if rot <= t:
            print(f"  t={t}: lô hỏng ngày {rot} đã hỏng -> bỏ"); continue
        k = min(c, rot - t)
        eaten += k
        print(f"  t={t}: ăn liên tục {k} quả của lô hỏng ngày {rot} (còn {c}) -> t={t + k}, tổng={eaten}")
        t += k
    return eaten

trace_jump([3, 0, 0, 0, 0, 2], [3, 0, 0, 0, 0, 2])
trace_jump([1, 2, 3, 5, 2], [3, 2, 1, 4, 2])

hết pha 1 (ngày 0..5): đã ăn 4, heap = [(7, 1)]
  t=6: ăn liên tục 1 quả của lô hỏng ngày 7 (còn 1) -> t=7, tổng=5
hết pha 1 (ngày 0..4): đã ăn 5, heap = [(6, 1), (7, 4)]
  t=5: ăn liên tục 1 quả của lô hỏng ngày 6 (còn 1) -> t=6, tổng=6
  t=6: ăn liên tục 1 quả của lô hỏng ngày 7 (còn 4) -> t=7, tổng=7


7

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def eatenApples(self, apples: List[int], days: List[int]) -> int:
        n = len(apples)
        h = []
        eaten = 0
        for t in range(n):
            if apples[t] > 0:
                heapq.heappush(h, (t + days[t], apples[t]))
            while h and h[0][0] <= t:
                heapq.heappop(h)
            if h:
                rot, c = heapq.heappop(h)
                eaten += 1
                if c > 1:
                    heapq.heappush(h, (rot, c - 1))
        t = n
        while h:
            rot, c = heapq.heappop(h)
            if rot <= t:
                continue
            k = min(c, rot - t)
            eaten += k
            t += k
        return eaten

sol = Solution()
for (apples, days), want in examples:
    got = sol.eatenApples(apples, days)
    print(got, "OK" if got == want else "SAI")

7 OK
5 OK


## 10. Vì sao thuật toán đúng

1. **Ăn mỗi khi có thể là tối ưu.** Nếu một lịch tối ưu bỏ trống ngày `t` trong khi có quả `y` ăn được, xét lần ăn `y` sau đó (nếu có) — dời việc ăn `y` về ngày `t`; nếu không ăn `y` thì thêm vào — số quả không giảm.

2. **Chọn quả hạn sớm nhất là tối ưu** (đổi chỗ, mục 6). Quy nạp theo ngày: tồn tại lịch tối ưu trùng EDF ở các ngày `0..t−1`; áp lập luận đổi chỗ cho ngày `t` mà không ảnh hưởng các ngày trước.

3. **Heap cài đúng EDF.** Mọi lô đã mọc đều được đẩy vào heap đúng ngày mọc. Lô bị bỏ khỏi đỉnh chỉ khi đã hỏng (`rot ≤ t`) hoặc hết quả — không bao giờ ăn được nữa. Khi đỉnh hợp lệ, nó có `rot` nhỏ nhất trong các lô hợp lệ (các lô hỏng/hết có thể còn nằm sâu trong heap nhưng có `rot` lớn hơn — chúng không ảnh hưởng tới đỉnh khi đỉnh hợp lệ, vì chúng sẽ bị lọc lúc nổi lên).

4. **Nhảy cóc đúng** (Cách 4): sau ngày `n` không có lô mới; EDF ăn liên tục lô đỉnh `k = min(c, rot − t)` ngày rồi lô đó hết hoặc hỏng — giống hệt mô phỏng từng ngày.

## 11. Phân tích Time / Space

Gọi `n ≤ 2·10⁴` là số ngày cây ra quả, `D = max(days) ≤ 2·10⁴`, `A = Σ apples`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Matching Kuhn | `O(A · Σ apples·days)` | `O(A·D)` | chỉ input tí hon |
| 2. EDF quét tuyến tính | `O(n · (n + D))` | `O(n)` | ~8·10⁸ worst |
| 3. EDF heap từng ngày | `O(n + D + n log n)` | `O(n)` | đủ nhanh |
| 4. Heap + nhảy cóc | `O(n log n)` | `O(n)` | **nộp bài** |

In [13]:
random.seed(1)
n = 2 * 10**4
apples = [random.randint(0, 2 * 10**4) for _ in range(n)]
days = [random.randint(1, 2 * 10**4) if a else 0 for a in apples]
for name, fn in [("heap", eaten_heap), ("jump", eaten_heap_jump), ("submit", Solution().eatenApples)]:
    t0 = time.perf_counter(); r = fn(apples, days)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.3f}s")

heap  : 39747  0.018s
jump  : 39747  0.019s
submit: 39747  0.019s


## 12. Bộ test

Đối chiếu với **ghép cặp cực đại** (định nghĩa gốc của bài) trên input nhỏ; input ngẫu nhiên tuân thủ ràng buộc `days[i] == 0 ⇔ apples[i] == 0`.

In [14]:
fns = [eaten_scan, eaten_heap, eaten_heap_jump, lambda a, d: Solution().eatenApples(a, d)]

for (apples, days), want in examples:
    assert eaten_matching(apples, days) == want
    for fn in fns:
        assert fn(apples, days) == want

edge = [
    (([0], [0]), 0),                         # không có táo
    (([5], [1]), 1),                         # hỏng ngay sau 1 ngày
    (([1], [100]), 1),                       # 1 quả, hạn dài
    (([20000], [20000]), 20000),             # ăn liên tục sau ngày n
    (([2, 1], [1, 10]), 2),                  # ngày 0 ăn lô 0, ngày 1 ăn lô 1
    (([0, 0, 3], [0, 0, 2]), 2),
]
for (apples, days), want in edge:
    if sum(apples) <= 50:
        assert eaten_matching(apples, days) == want, (apples, days)
    for fn in fns:
        if fn is eaten_scan and max(days) > 1000:
            continue
        assert fn(apples, days) == want, (fn, apples, days)

random.seed(2024)
for _ in range(400):
    n = random.randint(1, 8)
    apples = [random.choice([0, random.randint(1, 4)]) for _ in range(n)]
    days = [random.randint(1, 6) if a else 0 for a in apples]
    want = eaten_matching(apples, days)
    for fn in fns:
        assert fn(apples, days) == want, (fn, apples, days)
print("ví dụ + edge + 400 bộ ngẫu nhiên khớp ghép cặp cực đại")

ví dụ + edge + 400 bộ ngẫu nhiên khớp ghép cặp cực đại


In [15]:
random.seed(77)
for _ in range(60):
    n = random.randint(1, 300)
    apples = [random.choice([0, random.randint(1, 50)]) for _ in range(n)]
    days = [random.randint(1, 60) if a else 0 for a in apples]
    r = eaten_scan(apples, days)
    assert r == eaten_heap(apples, days) == eaten_heap_jump(apples, days)
print("n tới 300: EDF quét tuyến tính khớp hai bản heap")

n tới 300: EDF quét tuyến tính khớp hai bản heap


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Ăn quả mới nhất (LIFO) hoặc quả cũ nhất (FIFO theo ngày mọc)

Ngày mọc sớm không có nghĩa là hỏng sớm.

In [16]:
def wrong_fifo(apples, days):
    n = len(apples); batches = []; eaten = 0; t = 0
    T = max(i + d for i, d in enumerate(days))
    for t in range(T):
        if t < n and apples[t]:
            batches.append([t + days[t], apples[t]])
        batches = [b for b in batches if b[0] > t and b[1] > 0]
        if batches:
            batches[0][1] -= 1; eaten += 1         # SAI: ăn lô mọc sớm nhất
    return eaten

apples, days = [2, 1], [10, 1]
print("sai =", wrong_fifo(apples, days), "| đúng =", eaten_heap(apples, days))

sai = 2 | đúng = 3


Lô ngày 0 hạn dài (10 ngày) còn lô ngày 1 hỏng ngay sau ngày 1. FIFO ăn lô 0 ở ngày 1 ⇒ bỏ lỡ quả lô 1. **Sửa:** ưu tiên theo **ngày hỏng**.

### Lỗi 2 — Dừng ở ngày `n − 1`

Đề cho phép tiếp tục ăn sau `n` ngày.

In [17]:
def wrong_stop_at_n(apples, days):
    n = len(apples); h = []; eaten = 0
    for t in range(n):                      # SAI: không xử lý sau ngày n-1
        if apples[t] > 0:
            heapq.heappush(h, [t + days[t], apples[t]])
        while h and (h[0][0] <= t or h[0][1] == 0):
            heapq.heappop(h)
        if h:
            h[0][1] -= 1; eaten += 1
    return eaten

for (apples, days), want in examples:
    print("sai =", wrong_stop_at_n(apples, days), "| đúng =", want)

sai = 5 | đúng = 7
sai = 4 | đúng = 5


**Sửa:** tiếp tục vòng lặp khi heap còn lô (Cách 3) hoặc pha nhảy cóc (Cách 4).

### Lỗi 3 — Sai điều kiện hỏng: dùng `rot < t` thay cho `rot ≤ t`

Quả hỏng **vào** ngày `i + days[i]`, nên ở ngày `t = rot` đã không ăn được.

In [18]:
def wrong_rot_check(apples, days):
    n = len(apples); h = []; eaten = 0; t = 0
    while t < n or h:
        if t < n and apples[t] > 0:
            heapq.heappush(h, [t + days[t], apples[t]])
        while h and (h[0][0] < t or h[0][1] == 0):   # SAI: phải là <=
            heapq.heappop(h)
        if h:
            h[0][1] -= 1; eaten += 1
        t += 1
    return eaten

for (apples, days), want in examples:
    print("sai =", wrong_rot_check(apples, days), "| đúng =", want)

sai = 8 | đúng = 7
sai = 5 | đúng = 5


**Sửa:** lô hợp lệ ở ngày `t` ⇔ `t < rot` ⇔ bỏ khi `rot ≤ t`.

### Lỗi 4 — Nhảy cóc nhưng ăn **cả lô** mà quên giới hạn bởi hạn hỏng

Trong pha 2, viết `eaten += c` (ăn hết `c` quả) trong khi chỉ còn `rot − t` ngày trước khi lô hỏng.

In [19]:
def wrong_jump(apples, days):
    n = len(apples); h = []; eaten = 0
    for t in range(n):
        if apples[t] > 0:
            heapq.heappush(h, (t + days[t], apples[t]))
        while h and h[0][0] <= t:
            heapq.heappop(h)
        if h:
            rot, c = heapq.heappop(h); eaten += 1
            if c > 1: heapq.heappush(h, (rot, c - 1))
    t = n
    while h:
        rot, c = heapq.heappop(h)
        if rot <= t:
            continue
        eaten += c; t += c                   # SAI: phải là k = min(c, rot - t)
    return eaten

for (apples, days), want in examples:
    print("sai =", wrong_jump(apples, days), "| đúng =", want)

sai = 10 | đúng = 7
sai = 5 | đúng = 5


Ở ví dụ 1, sau ngày 4 lô hỏng ngày 7 còn 4 quả nhưng chỉ còn ngày 6 để ăn (ngày 5 dành cho lô hỏng ngày 6) — bản lỗi ăn cả 4. **Sửa:** `k = min(c, rot − t)`.

(Ghi chú: kiểm tra `if rot <= t: continue` trong pha 2 thực ra chỉ để rõ ràng — vì heap pop theo `rot` tăng dần và `t` sau mỗi lô không vượt `rot` của lô đó, nên `rot − t` không bao giờ âm; nhưng giữ dòng này giúp code an toàn khi sửa đổi.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mỗi đơn vị thời gian làm tối đa 1 việc, mỗi việc có **khoảng hợp lệ** `[release, deadline)`, tối đa số việc" ⇒ **EDF + min-heap theo deadline**.
- Lưu **lô** `(deadline, số lượng)` thay vì từng quả — tránh `Σ apples` phần tử.
- Xóa lười: lọc đỉnh heap mỗi bước.
- Khi không còn sự kiện mới, **nhảy thời gian** theo lô thay vì từng ngày.
- Cùng họ: *Maximum Number of Events That Can Be Attended* (1353) — gần như y hệt (mỗi sự kiện = 1 quả), *Course Schedule III* (630).

## 15. Biến thể của bài

1. **LeetCode 1353 — Maximum Number of Events That Can Be Attended:** mỗi sự kiện `[start, end]`, mỗi ngày dự 1 sự kiện ⇒ EDF y hệt (`rot = end + 1`).
2. **Ăn tối đa `k` quả mỗi ngày:** mỗi ngày pop/ăn tối đa `k` quả từ các lô hỏng sớm nhất.
3. **Trả về lịch ăn** (ngày → lô): ghi lại lô được ăn mỗi ngày.
4. **Mỗi quả có giá trị khác nhau, tối đa tổng giá trị:** không còn là EDF thuần — cần "Scheduling with deadlines and profits" (tham lam theo giá trị + DSU/heap ngược).

In [20]:
def max_events(events):
    """Biến thể 1 (LeetCode 1353) bằng cùng khung EDF."""
    events = sorted(events)
    h = []; i = 0; t = 0; res = 0; n = len(events)
    while i < n or h:
        if not h and t < events[i][0]:
            t = events[i][0]
        while i < n and events[i][0] <= t:
            heapq.heappush(h, events[i][1]); i += 1
        while h and h[0] < t:
            heapq.heappop(h)
        if h:
            heapq.heappop(h); res += 1
        t += 1
    return res

print(max_events([[1, 2], [2, 3], [3, 4]]), max_events([[1, 2], [2, 3], [3, 4], [1, 2]]))

3 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Matching | đồ thị quả–ngày, Kuhn | rất lớn | rất lớn | kiểm chứng định nghĩa |
| 2. EDF quét | mỗi ngày quét mọi lô | `O(n(n+D))` | `O(n)` | hiểu tham lam |
| 3. EDF heap từng ngày | min-heap theo hạn | `O(n + D + n log n)` | `O(n)` | an toàn, dễ viết |
| 4. Heap + nhảy cóc | ăn cả lô sau ngày `n` | `O(n log n)` | `O(n)` | **nộp bài** |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `eatenApples([2, 1, 10], [2, 10, 1])`?

**Bài 2 (tính tay):** trace Cách 3 cho `apples = [0, 3, 0, 1]`, `days = [0, 2, 0, 5]`.

**Bài 3 (điền TODO):** hoàn thiện `eating_schedule(apples, days)` trả về danh sách `(ngày, lô)` các quả được ăn theo EDF.

**Bài 4 (tự viết lại):** viết lại Cách 3 dùng heap của tuple bất biến `(rot, i)` và mảng `rem[i]` lưu số quả còn lại.

**Bài 5 (mini challenge):** biến thể ăn **tối đa `k` quả mỗi ngày**. Viết lời giải EDF và kiểm chứng bằng ghép cặp (mỗi ngày nhân bản thành `k` đỉnh).

In [21]:
def eating_schedule(apples, days):
    # TODO: như Cách 3, mỗi khi ăn thì ghi (t, chỉ số lô)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Lô 2 (10 quả) hỏng ngay sau ngày 2. (2) Ngày 0 ăn lô 0, ngày 1 ăn lô 0 (hỏng ở ngày 2), ngày 2 ăn lô 2. (3) Sau đó lô 1 (hạn tới ngày 10) ăn tiếp ngày 3 → không còn (chỉ 1 quả). Đáp án `4`.

**Bài 2:** (1) Ngày 0 không có gì. (2) Ngày 1, 2 ăn lô 1 (hỏng ở ngày 3). (3) Ngày 3 ăn lô 3; tổng `3`.

**Bài 3:** (1) Heap lưu `[rot, count, i]`. (2) Khi ăn: `out.append((t, h[0][2]))`. (3) Dùng mô phỏng từng ngày (không nhảy cóc) để có đủ từng ngày.

**Bài 4:** (1) `heappush(h, (t + days[t], t))`. (2) Lọc khi `rot ≤ t` hoặc `rem[i] == 0`. (3) Ăn: `rem[i] −= 1`.

**Bài 5:** (1) Mỗi ngày lặp tối đa `k` lần "lọc đỉnh, ăn một quả". (2) Matching: đỉnh phải là `(ngày, slot)`. (3) So sánh trên input nhỏ ngẫu nhiên.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert Solution().eatenApples([2, 1, 10], [2, 10, 1]) == 4 == eaten_matching([2, 1, 10], [2, 10, 1])

# Bài 2
assert eaten_heap([0, 3, 0, 1], [0, 2, 0, 5]) == 3

# Bài 3
def eating_schedule_ans(apples, days):
    n = len(apples); h = []; out = []; t = 0
    while t < n or h:
        if t < n and apples[t] > 0:
            heapq.heappush(h, [t + days[t], apples[t], t])
        while h and (h[0][0] <= t or h[0][1] == 0):
            heapq.heappop(h)
        if h:
            h[0][1] -= 1; out.append((t, h[0][2]))
        t += 1
    return out

sch = eating_schedule_ans([1, 2, 3, 5, 2], [3, 2, 1, 4, 2])
assert len(sch) == 7 and len({d for d, _ in sch}) == 7
assert all(i <= d < i + [3, 2, 1, 4, 2][i] for d, i in sch)     # mỗi quả ăn đúng hạn

# Bài 4
def eaten_heap_rem(apples, days):
    n = len(apples); rem = list(apples); h = []; eaten = 0; t = 0
    while t < n or h:
        if t < n and apples[t] > 0:
            heapq.heappush(h, (t + days[t], t))
        while h and (h[0][0] <= t or rem[h[0][1]] == 0):
            heapq.heappop(h)
        if h:
            rem[h[0][1]] -= 1; eaten += 1
        t += 1
    return eaten

random.seed(51)
for _ in range(300):
    n = random.randint(1, 30)
    apples = [random.choice([0, random.randint(1, 10)]) for _ in range(n)]
    days = [random.randint(1, 12) if a else 0 for a in apples]
    assert eaten_heap_rem(apples, days) == eaten_heap_jump(apples, days)
print("Bài 1-4: OK")

Bài 1-4: OK


In [23]:
# Bài 5 — tối đa k quả mỗi ngày
def eaten_k_per_day(apples, days, k):
    n = len(apples); h = []; eaten = 0; t = 0
    while t < n or h:
        if t < n and apples[t] > 0:
            heapq.heappush(h, [t + days[t], apples[t]])
        for _ in range(k):
            while h and (h[0][0] <= t or h[0][1] == 0):
                heapq.heappop(h)
            if not h:
                break
            h[0][1] -= 1; eaten += 1
        t += 1
    return eaten

def eaten_matching_k(apples, days, k):
    T = max((i + d for i, d in enumerate(days)), default=0)
    adj = []
    for i, (a, d) in enumerate(zip(apples, days)):
        for _ in range(a):
            adj.append([day * k + s for day in range(i, i + d) for s in range(k)])
    return kuhn(adj, T * k)

random.seed(52)
for _ in range(300):
    n = random.randint(1, 6); k = random.randint(1, 3)
    apples = [random.choice([0, random.randint(1, 5)]) for _ in range(n)]
    days = [random.randint(1, 4) if a else 0 for a in apples]
    assert eaten_k_per_day(apples, days, k) == eaten_matching_k(apples, days, k), (apples, days, k)
assert eaten_k_per_day([1, 2, 3, 5, 2], [3, 2, 1, 4, 2], 1) == 7
print("Bài 5: EDF k quả/ngày khớp ghép cặp; ví dụ 1 với k=2 ->", eaten_k_per_day([1, 2, 3, 5, 2], [3, 2, 1, 4, 2], 2))

Bài 5: EDF k quả/ngày khớp ghép cặp; ví dụ 1 với k=2 -> 12


## 20. Template nhớ nhanh + Checklist

```python
# EDF: mỗi ngày ăn quả hỏng sớm nhất; heap (ngày_hỏng, số_quả)
h = []; eaten = 0
for t in range(n):
    if apples[t]: heappush(h, (t + days[t], apples[t]))
    while h and h[0][0] <= t: heappop(h)          # bỏ lô đã hỏng
    if h:
        rot, c = heappop(h); eaten += 1
        if c > 1: heappush(h, (rot, c - 1))
t = n
while h:                                           # sau ngày n: ăn cả lô
    rot, c = heappop(h)
    if rot <= t: continue
    k = min(c, rot - t); eaten += k; t += k
```

**Checklist trước khi submit:**
- [ ] Ưu tiên theo **ngày hỏng** `i + days[i]`, không theo ngày mọc.
- [ ] Hỏng khi `rot ≤ t` (ngày `rot` đã không ăn được).
- [ ] Tiếp tục ăn **sau** ngày `n − 1`.
- [ ] Bỏ qua ngày `apples[i] == 0`.
- [ ] Pha nhảy cóc: `k = min(c, rot − t)`, bỏ qua lô đã hỏng.